1.Load and validate the dataset

In [3]:

import pandas as pd

df = pd.read_csv("heart_failure_clinical_records_dataset.csv")

print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nMortality outcome distribution:")
print(df["DEATH_EVENT"].value_counts().sort_index())

print("\nMortality outcome percentages:")
print(
    (df["DEATH_EVENT"].value_counts(normalize=True).sort_index() * 100)
    .round(2)
)


Dataset shape: (299, 13)

Column names:
['age', 'anaemia', 'creatinine_phosphokinase', 'diabetes', 'ejection_fraction', 'high_blood_pressure', 'platelets', 'serum_creatinine', 'serum_sodium', 'sex', 'smoking', 'time', 'DEATH_EVENT']

Missing values:
age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0
dtype: int64

Duplicate rows: 0

Mortality outcome distribution:
DEATH_EVENT
0    203
1     96
Name: count, dtype: int64

Mortality outcome percentages:
DEATH_EVENT
0    67.89
1    32.11
Name: proportion, dtype: float64


2.Split the data

In [4]:

from sklearn.model_selection import train_test_split


y = df["DEATH_EVENT"]
# Exclude the target and follow-up time
X = df.drop(columns=["DEATH_EVENT", "time"])


X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20, random_state=42,stratify=y)

# Define the two feature sets
selected_features = ["serum_creatinine", "ejection_fraction"]

X_train_two = X_train[selected_features]
X_test_two = X_test[selected_features]

# Display the split information
print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))

print("\nTraining outcome distribution:")
print(y_train.value_counts().sort_index())

print("\nTesting outcome distribution:")
print(y_test.value_counts().sort_index())

print("\nModel A training shape:", X_train.shape)
print("Model A testing shape:", X_test.shape)

print("\nModel B training shape:", X_train_two.shape)
print("Model B testing shape:", X_test_two.shape)



Training records: 239
Testing records: 60

Training outcome distribution:
DEATH_EVENT
0    162
1     77
Name: count, dtype: int64

Testing outcome distribution:
DEATH_EVENT
0    41
1    19
Name: count, dtype: int64

Model A training shape: (239, 11)
Model A testing shape: (60, 11)

Model B training shape: (239, 2)
Model B testing shape: (60, 2)


3.Train both baseline models

In [5]:

from sklearn.ensemble import RandomForestClassifier

# Model A: use all 11 features
model_a = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
model_a.fit(X_train, y_train)

# Model B: use only the two selected features
model_b = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")
model_b.fit(X_train_two, y_train)

print("\nBoth Random Forest models have been trained successfully.")



Both Random Forest models have been trained successfully.


4.Evaluate both models

In [6]:
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    roc_auc_score,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt
from pathlib import Path


# Create a folder for the output figures
output_folder = Path("results")
output_folder.mkdir(exist_ok=True)


def evaluate_model(model, X_test_data, y_test_data, model_name, filename):
    # Predict the outcome using unseen test data
    predictions = model.predict(X_test_data)

    # Probability of death (DEATH_EVENT = 1)
    probabilities = model.predict_proba(X_test_data)[:, 1]

    # Confusion matrix: [[TN, FP], [FN, TP]]
    tn, fp, fn, tp = confusion_matrix(
        y_test_data,
        predictions,
        labels=[0, 1]
    ).ravel()

    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test_data, predictions)
    sensitivity = recall_score(
        y_test_data, predictions, zero_division=0
    )
    specificity = tn / (tn + fp) if (tn + fp) else 0
    precision = precision_score(
        y_test_data, predictions, zero_division=0
    )
    f1 = f1_score(
        y_test_data, predictions, zero_division=0
    )
    mcc = matthews_corrcoef(y_test_data, predictions)
    roc_auc = roc_auc_score(y_test_data, probabilities)

    # Display the results
    print(f"\n{'=' * 45}")
    print(model_name)
    print(f"{'=' * 45}")

    print("\nConfusion matrix:")
    print([[tn, fp], [fn, tp]])

    print(f"True negatives (TN): {tn}")
    print(f"False positives (FP): {fp}")
    print(f"False negatives (FN): {fn}")
    print(f"True positives (TP): {tp}")

    print(f"\nAccuracy:     {accuracy:.3f}")
    print(f"Sensitivity:  {sensitivity:.3f}")
    print(f"Specificity:  {specificity:.3f}")
    print(f"Precision:    {precision:.3f}")
    print(f"F1-score:     {f1:.3f}")
    print(f"MCC:          {mcc:.3f}")
    print(f"ROC-AUC:      {roc_auc:.3f}")

    # Create and save the confusion matrix figure
    display = ConfusionMatrixDisplay.from_predictions(
        y_test_data,
        predictions,
        labels=[0, 1],
        display_labels=["Survived (0)", "Died (1)"],
        cmap="Blues",
        values_format="d"
    )

    display.ax_.set_title(model_name)
    display.figure_.tight_layout()

    figure_path = output_folder / filename
    display.figure_.savefig(
        figure_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.close(display.figure_)
    print(f"Confusion matrix saved to: {figure_path}")

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "F1-score": f1,
        "MCC": mcc,
        "ROC-AUC": roc_auc
    }


# Evaluate both models on the same test patients
results_a = evaluate_model(
    model_a,
    X_test,
    y_test,
    "Model A: Random Forest with 11 Features",
    "figure3_confusion_matrix_model_a.png"
)

results_b = evaluate_model(
    model_b,
    X_test_two,
    y_test,
    "Model B: Random Forest with Two Features",
    "figure4_confusion_matrix_model_b.png"
)



Model A: Random Forest with 11 Features

Confusion matrix:
[[np.int64(36), np.int64(5)], [np.int64(13), np.int64(6)]]
True negatives (TN): 36
False positives (FP): 5
False negatives (FN): 13
True positives (TP): 6

Accuracy:     0.700
Sensitivity:  0.316
Specificity:  0.878
Precision:    0.545
F1-score:     0.400
MCC:          0.233
ROC-AUC:      0.800
Confusion matrix saved to: results\figure3_confusion_matrix_model_a.png

Model B: Random Forest with Two Features

Confusion matrix:
[[np.int64(35), np.int64(6)], [np.int64(11), np.int64(8)]]
True negatives (TN): 35
False positives (FP): 6
False negatives (FN): 11
True positives (TP): 8

Accuracy:     0.717
Sensitivity:  0.421
Specificity:  0.854
Precision:    0.571
F1-score:     0.485
MCC:          0.302
ROC-AUC:      0.731
Confusion matrix saved to: results\figure4_confusion_matrix_model_b.png


5.comparison table

In [7]:

results_df = pd.DataFrame([results_a, results_b])

print("\n\nFINAL MODEL COMPARISON")
print(results_df.round(3).to_string(index=False))

# Save the results for your coursework
results_df.to_csv("baseline_model_comparison.csv", index=False)

print("\nResults saved to baseline_model_comparison.csv")




FINAL MODEL COMPARISON
                                   Model  Accuracy  Sensitivity  Specificity  Precision  F1-score   MCC  ROC-AUC
 Model A: Random Forest with 11 Features     0.700        0.316        0.878      0.545     0.400 0.233    0.800
Model B: Random Forest with Two Features     0.717        0.421        0.854      0.571     0.485 0.302    0.731

Results saved to baseline_model_comparison.csv
